# NT120: MULTI-TASK VIETNAMESE SENTIMENT AND TOXICITY CLASSIFICATION
## Architecture: BaMiBERT (Transformer Encoder with Feature Fusion and Attention Pooling)

### Project Specification Overview:
- **Task 1 (Sentiment Analysis):** Negative (-1), Neutral (0), Positive (1)
- **Task 2 (Toxicity Detection):** Clean (0), Toxic (1), Very Toxic (2), Extremely Toxic (3)

### Core Pipeline Components:
1. **Minimalist Preprocessing:** Unicode NFC normalization, control character filtering, dynamic batch padding.
2. **Multi-Layer Feature Fusion:** Learned combination of hidden representations across final encoder layers.
3. **Masked Attention Pooling:** Attention-weighted sentence representation with explicit pad masking.
4. **Task-Specific Heads:** Independent projection modules with Multi-Sample Dropout (5 parallel streams).
5. **Imbalance & Ambiguity Regularization:** Logit-Adjusted Cross-Entropy for Toxicity; Label-Smoothed Cross-Entropy for Sentiment.
6. **Optimization:** Differential learning rates (backbone vs heads), AdamW optimizer, linear warmup with cosine decay, and composite validation Macro-F1 early stopping.

--- 
## 1. Hardware and Environment Verification

In [ ]:
import torch

print('PyTorch Version:', torch.__version__)
if torch.cuda.is_available():
    print('GPU Device:', torch.cuda.get_device_name(0))
    print('CUDA Available: True')
    !nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv
else:
    print('WARNING: CUDA is not available. Please navigate to Runtime -> Change runtime type -> T4 GPU.')


--- 
## 2. Repository Setup and Dependency Installation

In [ ]:
import os

PROJECT_DIR = '/content/NT120-BaMiBERT'
REPO_URL = 'https://github.com/TiLDyiou/NT120-BaMiBERT.git'

if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    # Pull latest updates if already cloned
    !git -C {PROJECT_DIR} pull origin main

%cd {PROJECT_DIR}

# Install required dependencies
!pip install -q transformers accelerate sentencepiece scikit-learn pandas numpy
print('Environment setup completed successfully.')


--- 
## 3. Dataset Distribution and Cross-Tabulation Analysis

In [ ]:
import pandas as pd

train_df = pd.read_csv('data/train/train.csv')
valid_df = pd.read_csv('data/valid/valid.csv')
chal_df = pd.read_csv('data/challenge/challenge.csv')

print(f'Train Samples:      {len(train_df):,}')
print(f'Validation Samples: {len(valid_df):,}')
print(f'Challenge Samples:  {len(chal_df):,}')

print('\n[Sentiment Distribution - Train]')
print(train_df['Sentiment'].value_counts())

print('\n[Toxicity Distribution - Train]')
print(train_df['Toxicity'].value_counts())

print('\n[Cross-Tabulation: Sentiment vs Toxicity]')
display(pd.crosstab(train_df['Sentiment'], train_df['Toxicity'], margins=True))


--- 
## 4. Multi-Task Model Training

Execute training using `vinai/phobert-base-v2` as the transformer backbone.
- Pre-split deduplication is automatically applied.
- Toxicity prior frequencies are extracted to initialize Logit Adjustment (tau=1.0).
- Optimal checkpoint is evaluated on validation set and saved to `checkpoints/best_model.pt`.

In [ ]:
!python train.py \
    --model_name 'vinai/phobert-base-v2' \
    --batch_size 32 \
    --epochs 6 \
    --lr_backbone 2e-5 \
    --lr_heads 1e-4 \
    --tau 1.0 \
    --label_smoothing 0.1 \
    --output_dir 'checkpoints'


--- 
## 5. Quantitative Evaluation on Challenge Dataset

In [ ]:
!python evaluate.py \
    --data_path 'data/challenge/challenge.csv' \
    --checkpoint 'checkpoints/best_model.pt' \
    --output_file 'challenge_predictions.csv'


--- 
## 6. Confusion Matrix and Error Visualization

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

PRED_FILE = 'challenge_predictions.csv'

if not os.path.exists(PRED_FILE):
    print(f'Error: File {PRED_FILE} not found. Please complete Step 5 first.')
else:
    results_df = pd.read_csv(PRED_FILE)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # 1. Sentiment Confusion Matrix
    sent_map = {-1: 0, 0: 1, 1: 2}
    y_true_s = [sent_map[int(x)] for x in results_df['Sentiment']]
    y_pred_s = results_df['Pred_Sentiment_Raw'].astype(int)
    cm_s = confusion_matrix(y_true_s, y_pred_s)

    sns.heatmap(
        cm_s, annot=True, fmt='d', cmap='Blues', ax=axes[0],
        xticklabels=['Negative (-1)', 'Neutral (0)', 'Positive (1)'],
        yticklabels=['Negative (-1)', 'Neutral (0)', 'Positive (1)']
    )
    axes[0].set_title('Confusion Matrix - Sentiment Analysis')
    axes[0].set_xlabel('Predicted Label')
    axes[0].set_ylabel('Ground Truth Label')

    # 2. Toxicity Confusion Matrix
    y_true_t = results_df['Toxicity'].astype(int)
    y_pred_t = results_df['Pred_Toxicity'].astype(int)
    cm_t = confusion_matrix(y_true_t, y_pred_t)

    sns.heatmap(
        cm_t, annot=True, fmt='d', cmap='Reds', ax=axes[1],
        xticklabels=['Clean (0)', 'Toxic (1)', 'Very Toxic (2)', 'Extremely (3)'],
        yticklabels=['Clean (0)', 'Toxic (1)', 'Very Toxic (2)', 'Extremely (3)']
    )
    axes[1].set_title('Confusion Matrix - Toxicity Detection')
    axes[1].set_xlabel('Predicted Label')
    axes[1].set_ylabel('Ground Truth Label')

    plt.tight_layout()
    plt.show()


--- 
## 7. Interactive Single-Sample Inference

In [ ]:
import os
import torch
from transformers import AutoTokenizer
from src.dataset import clean_text_minimalist, ID_TO_SENTIMENT, ID_TO_TOXICITY
from src.model import BaMiBERT

CHECKPOINT_PATH = 'checkpoints/best_model.pt'
MODEL_NAME = 'vinai/phobert-base-v2'

if not os.path.exists(CHECKPOINT_PATH):
    print(f'Error: Checkpoint {CHECKPOINT_PATH} not found. Please train the model in Step 4.')
else:
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)

    model = BaMiBERT(model_name=MODEL_NAME, pretrained=False)
    state_dict = torch.load(CHECKPOINT_PATH, map_location=device)
    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()

    def predict_comment(text: str):
        cleaned = clean_text_minimalist(text)
        inputs = tokenizer(cleaned, return_tensors='pt', truncation=True, max_length=256)
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(inputs['input_ids'], inputs['attention_mask'])

        s_probs = torch.softmax(outputs['sentiment_logits'], dim=-1)[0].cpu().numpy()
        t_probs = torch.softmax(outputs['toxicity_logits'], dim=-1)[0].cpu().numpy()

        s_id = s_probs.argmax()
        t_id = t_probs.argmax()

        s_label = ID_TO_SENTIMENT[s_id]
        t_label = ID_TO_TOXICITY[t_id]

        s_names = {-1: 'Negative (-1)', 0: 'Neutral (0)', 1: 'Positive (1)'}
        t_names = {0: 'Clean (0)', 1: 'Toxic (1)', 2: 'Very Toxic (2)', 3: 'Extremely Toxic (3)'}

        print('-' * 70)
        print(f'Input Text:  "{text}"')
        print(f'Sentiment:   {s_names[s_label]} (Confidence: {s_probs[s_id]*100:.2f}%)')
        print(f'Toxicity:    {t_names[t_label]} (Confidence: {t_probs[t_id]*100:.2f}%)')
        print('-' * 70)

    # Benchmark qualitative validation examples
    predict_comment('Dịch vụ phục vụ rất chậm, tôi hoàn toàn thất vọng về quán hôm nay.')
    predict_comment('Đm đỉnh vãi nồi, quá đẳng cấp các ông ơi :))) !!!')
    predict_comment('Cút đi con chó ngu học, biến khỏi mắt tao ngay!')


--- 
## 8. Export Checkpoints and Result Artifacts

In [ ]:
from google.colab import files

export_files = ['challenge_predictions.csv', 'checkpoints/best_metrics.json']
for f_path in export_files:
    if os.path.exists(f_path):
        print(f'Downloading {f_path}...')
        files.download(f_path)
    else:
        print(f'Artifact {f_path} does not exist yet.')
